# Walkthrough — GOPT: Generalizable Online 3D Bin Packing

Xiong et al. (2024). Official code: https://github.com/Xiong5Heng/GOPT

This notebook connects each paper section to the code in `src/` with **runnable sanity checks** and
short theory cells. Everything runs on CPU in seconds at toy scale (small `d_model`, `N`, `batch`) —
the architecture is identical to the paper, only the sizes shrink. See `REPRODUCTION_NOTES.md` for
flagged assumptions and `PAPER_GUIDE.md` for the section-by-section narrative.

Pipeline: **Placement Generator (EMS state) → embeddings → Packing Transformer → actor/critic → PPO + GAE**.

## 1. Setup

Add the repo root to the path (modules import as `src.*`) and build a **toy** `ModelConfig`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root, so `import src.model` works
import torch
from src.utils import ModelConfig

# Toy dimensions for the walkthrough (paper: d_model=128, n_layers=3, n_heads=8, N=80).
cfg = ModelConfig(d_model=32, n_layers=2, n_heads=4, d_ff=32, max_ems=8)
B, N = 2, cfg.max_ems  # batch size, EMS sequence length
torch.manual_seed(0)
print(f'toy config: d_model={cfg.d_model}, n_layers={cfg.n_layers}, n_heads={cfg.n_heads}, N={N}')

## 2. The MDP state — Placement Generator & EMS (§III-B, §III-C)

> "Each EMS can be defined by its FLB vertex and the corresponding opposite vertex ... The resulting
> 6-dimensional vector is normalized to [0,1], regardless of the dimensions of the bin." — §III-B

**Background.** Rather than letting the agent pick any `(x,y,z)`, a heuristic proposes a fixed-length
set of `N` **Empty Maximal Spaces** (largest empty boxes). Each EMS is a 6-D box (FLB + opposite
vertex), **normalized to [0,1]** — so the policy never sees absolute bin size, which is what lets one
policy generalize across bins. The item is a **2×3 matrix** (rows = the 0° and 90° orientations). The
action space is `2·N` (orientation × EMS) — independent of bin dimensions (§III-C).

In [ ]:
from src.data import BinPackingEnv

env = BinPackingEnv(L=10, W=10, H=10, max_ems=N)
ems_features, item_features, action_mask = env.reset()
print('EMS features  :', tuple(ems_features.shape), '-> (N, 6) box vectors')
print('item features :', tuple(item_features.shape), '-> (2, 3) two orientations')
print('action mask   :', tuple(action_mask.shape), '-> (2, N) feasible (orientation, EMS) pairs')
assert ems_features.shape == (N, 6) and item_features.shape == (2, 3)
assert action_mask.shape == (2, N) and action_mask.dtype == torch.bool
# Action space size is 2*N, independent of bin L/W/H (the generalization property, §III-C):
assert 2 * N == action_mask.numel()
print('\u2713 state shapes match the paper; |A| = 2N =', 2 * N)

### Reward intuition (§III-C)
The reward is the **volume fraction** an item adds: `r_t = (l·w·h)/(L·W·H)`. It is *dense* (every
placement gives signal), which §IV-D shows trains better than a sparse terminal reward.

In [ ]:
_, reward, done, info = env.step(action_idx=0)
print('reward for one placement =', reward, '| decoded action =', info)
assert 0.0 <= reward <= 1.0, 'a volume fraction must lie in [0,1]'
print('\u2713 reward is a volume fraction in [0,1]')

## 3. The Packing Transformer block — self + cross attention (§III-D)

> "each containing two self-attention layers, one bi-directional cross-attention layer, and four MLP
> blocks ... Residual connections and layer normalization are applied after each layer." — §III-D

**Background.** *Self*-attention links EMSs to EMSs (and item rows to each other) — intra-set
structure. *Cross*-attention links the two sets **both ways** (EMS→item, item→EMS): "does this space
suit this item?". Each sublayer is wrapped in a residual + post-LayerNorm: `x ← Norm(x + sublayer(x))`.

In [ ]:
from src.model import EncoderBlock, MLP

block = EncoderBlock(cfg)
ems = torch.randn(B, N, cfg.d_model)    # (B, N, d) — embedded EMSs
item = torch.randn(B, 2, cfg.d_model)   # (B, 2, d) — embedded item rows
ems_out, item_out = block(ems, item)
print('EMS :', tuple(ems.shape), '->', tuple(ems_out.shape))
print('item:', tuple(item.shape), '->', tuple(item_out.shape))
assert ems_out.shape == ems.shape and item_out.shape == item.shape, 'attention preserves shape'
print('\u2713 a block fuses EMS\u2194item features while preserving shapes')

### What would happen without cross-attention?
If you drop the EMS↔item cross-attention, the item features never "see" the available spaces — the
actor would score placements without knowing whether the item fits. §IV-D's ablation (GOPT w/o PT)
shows exactly this: a large drop in space utilization. We confirm cross-attention actually moves the
features (item_out differs from item_in).

In [ ]:
assert not torch.allclose(item_out, item), 'cross/self-attention must transform the item features'
print('\u2713 item features are updated by attention to the EMS set (not a no-op)')

## 4. Full model — actor score map + critic value (§III-D)

> "both the EMS and item features are processed through an MLP, and the results are multiplied to
> compute a score map of actions ... followed by an element-wise multiplication with the action
> mask to eliminate infeasible actions." — §III-D

The actor scores action `(orientation o, EMS j)` as the dot product of item-row `o` with EMS `j` →
`(2, N)` logits, flattened to `2N`. The critic mean-pools the sequences into one `V(s)`.

In [ ]:
from src.model import GOPTModel

model = GOPTModel(cfg)
ems_in = torch.rand(B, N, cfg.ems_dim)     # (B, N, 6)
item_in = torch.rand(B, 2, cfg.item_dim)   # (B, 2, 3)
mask = torch.ones(B, 2, N, dtype=torch.bool)
mask[:, 1, N - 1] = False                  # mark one (orientation=90°, last EMS) infeasible

logits, value = model(ems_in, item_in, mask)
print('action logits:', tuple(logits.shape), '-> (B, 2N)')
print('state value  :', tuple(value.shape), '-> (B, 1)')
assert logits.shape == (B, 2 * N) and value.shape == (B, 1)
# The masked action must be -inf so softmax gives it probability 0 (§III-C):
flat_masked = (~mask).view(B, -1)
assert torch.isinf(logits[flat_masked]).all(), 'infeasible actions must be -inf'
probs = torch.softmax(logits, dim=-1)
assert torch.allclose(probs.sum(-1), torch.ones(B), atol=1e-5)
assert (probs[flat_masked] == 0).all(), 'infeasible actions must get probability 0'
print('\u2713 forward pass OK; masking drives infeasible actions to probability 0')
print('  params:', sum(p.numel() for p in model.parameters()))

## 5. PPO loss (§III-E, §IV-A)

> "optimize the following objective ... L^CLIP = E[min(p_t(θ)Â_t, clip(p_t(θ), 1−ε, 1+ε)Â_t)]" — §III-E

Three terms: clipped policy loss + value MSE + entropy bonus, with `c_value=0.5, c_entropy=0.001,
ε=0.3` (§IV-A). We compute it on a sampled action distribution from the model above.

In [ ]:
from torch.distributions import Categorical
from src.loss import ppo_loss

dist = Categorical(logits=logits)
actions = dist.sample()                # (B,) sampled placements (never an infeasible one)
assert not flat_masked.gather(1, actions[:, None]).any(), 'sampler must avoid masked actions'
new_lp = dist.log_prob(actions)
ent = dist.entropy()
# Dummy rollout targets (a real run gets these from collection + GAE):
old_lp = new_lp.detach()
adv = torch.randn(B)
ret = torch.randn(B)
loss, pl, vl, el = ppo_loss(new_lp, old_lp, adv, value.squeeze(-1), ret, ent,
                            clip_ratio=0.3, c_value=0.5, c_entropy=0.001)
print(f'total={loss.item():.4f}  policy={pl.item():.4f}  value={vl.item():.4f}  entropy={el.item():.4f}')
assert torch.isfinite(loss), 'loss must be finite'
print('\u2713 PPO loss computes and is finite')

## 6. GAE — advantage estimation (§III-E)

GAE turns a sequence of rewards + critic values into low-variance advantages `Â_t` via an
exponentially-weighted (`λ_GAE`) sum of one-step TD errors. GOPT uses `λ_GAE=0.96, γ=1` (§IV-A).

In [ ]:
from src.loss import compute_gae

T = 5  # toy rollout length
rewards = torch.rand(T, B) * 0.05      # small per-step volume gains
values = torch.randn(T, B)
dones = torch.zeros(T, B); dones[-1] = 1.0  # episode ends at the last step
last_value = torch.zeros(B)
adv_gae, returns = compute_gae(rewards, values, dones, last_value, gamma=1.0, gae_lambda=0.96)
print('advantages shape:', tuple(adv_gae.shape), '| returns shape:', tuple(returns.shape))
assert adv_gae.shape == (T, B) and returns.shape == (T, B)
assert torch.allclose(returns, adv_gae + values), 'returns must equal advantages + values'
print('\u2713 GAE returns = advantages + values (critic targets)')

## 7. One training step — gradients flow

Verify a backward pass produces finite gradients for every parameter (the wiring works end to end).

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=7e-5)  # §IV-A base LR
opt.zero_grad(); loss.backward(); opt.step()
missing = [n for n, p in model.named_parameters() if p.grad is None]
assert not missing, f'no gradient for: {missing}'
assert all(torch.isfinite(p.grad).all() for p in model.parameters()), 'all grads finite'
print('\u2713 one optimizer step done; all parameters received finite gradients')

## 8. Common pitfalls

1. **EMS normalization is the generalization mechanism (§III-B).** EMS vectors are normalized to
   [0,1] so the policy never sees absolute bin size. Feeding raw coordinates breaks cross-bin
   generalization — the paper's headline claim.
2. **Action = (orientation, EMS) jointly (§III-C).** `|A| = 2N`. The flat index decodes as
   `orientation = idx // N`, `ems = idx % N`. Mixing up the flatten order silently corrupts actions.
3. **Mask in logit space, not probability space.** Set infeasible logits to `-inf` (not multiply
   probabilities by 0 after softmax) so the distribution renormalizes correctly (§III-C).
4. **Critic pooling is `[UNSPECIFIED]` (§III-D).** We mean-pool; the paper doesn't say. Different
   pooling (max, attention-pool) can change value estimates and thus advantages.
5. **Head count is a guess.** §III-D never states the number of attention heads; we use 8
   (REPRODUCTION_NOTES). It must divide `d_model`.
6. **Dense vs terminal reward (§IV-D).** Use the step-wise volume reward; the terminal-only reward
   trains markedly worse.
7. **This is a scaffold.** `src/data.py` returns correctly-shaped but placeholder states — the real
   EMS extraction + stability/heightmap update are TODOs. Wire a real env before expecting learning.

## Further reading
- Schulman et al. (2017) — *Proximal Policy Optimization*.
- Schulman et al. (2016) — *Generalized Advantage Estimation*.
- Vaswani et al. (2017) — *Attention Is All You Need* (self/cross-attention).
- Zhao et al. (2021/2022) — CNN / constrained-MDP online 3D-BPP (baseline lineage).

See `PAPER_GUIDE.md` for the full narrative and `REPRODUCTION_NOTES.md` for every flagged assumption.